In [20]:
import os
import certifi
import requests

from dotenv import load_dotenv, find_dotenv
from langsmith import Client
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain.agents import create_agent
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver




In [21]:
_= load_dotenv(find_dotenv())
os.environ["SSL_CERT_FILE"] = certifi.where()
gemini_api_key = os.environ["GEMINI_API_KEY"]
tavily_api_key = os.environ["TAVILY_API_KEY"]
gemini_model = os.environ["GEMINI_MODEL"]
weather_api_key = os.environ["WEATHER_API_KEY"]


In [22]:
import time

EPA_AQI_LABELS = {
    "1": "Good",
    "2": "Moderate",
    "3": "Unhealthy for Sensitive Groups",
    "4": "Unhealthy",
    "5": "Very Unhealthy",
    "6": "Hazardous",
}


@tool
def get_weather(city: str) -> str:
    """Get the current weather and air quality (AQI, PM2.5, PM10) for a given city."""
    url = "http://api.weatherstack.com/current"
    params = {"access_key": weather_api_key, "query": city.strip()}

    response = requests.get(url, params=params)
    if response.status_code == 429:  # rate limited: wait and retry once
        time.sleep(2)
        response = requests.get(url, params=params)
    data = response.json()

    # Weatherstack can return errors with HTTP 200, so check the body instead
    if "current" not in data:
        error = data.get("error", {}).get("info", "unknown error")
        return f"Could not fetch weather data for {city}: {error}"

    current = data["current"]
    result = (
        f"City: {city}\n"
        f"Temperature: {current['temperature']}°C\n"
        f"Weather: {current['weather_descriptions'][0]}\n"
        f"Humidity: {current['humidity']}%"
    )

    aq = current.get("air_quality")
    if aq:
        epa_index = aq.get("us-epa-index")
        label = EPA_AQI_LABELS.get(epa_index, "Unknown")
        result += (
            f"\nAir Quality (US EPA index): {epa_index} - {label}\n"
            f"PM2.5: {aq.get('pm2_5')} µg/m³\n"
            f"PM10: {aq.get('pm10')} µg/m³"
        )

    return result

In [23]:
search_tool = TavilySearchResults(api_key=tavily_api_key,max_results=2)

In [24]:
result = search_tool.invoke("What is LangChain")

In [25]:
result

[{'title': 'What Is LangChain? | IBM',
  'url': 'https://www.ibm.com/think/topics/langchain',
  'content': 'LangChain is essentially a library of abstractions for Python and Javascript, representing common steps and concepts necessary to work with language models. These modular components—like functions and object classes—serve as the building blocks of generative AI programs. They can be “chained” together to create applications, minimizing the amount of code and fine understanding required to execute complex NLP tasks. Though LangChain’s abstracted approach may limit the extent to which an expert programmer can finely customize an application, it empowers specialists and newcomers alike to quickly experiment and prototype.\n\n### Importing language models [...] LangChain serves as a generic interface for nearly any LLM, providing a centralized development environment to build LLM applications and integrate them with external data sources and software workflows. LangChain’s module-bas

In [26]:
llm = ChatGoogleGenerativeAI(api_key=gemini_api_key, model=gemini_model, temperature=0)

In [27]:
tools = [search_tool, get_weather]

In [28]:
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=(
        "You are a helpful assistant. Use the tools when you need real data. "
        "Always reply in English."
    ),
    checkpointer = InMemorySaver()
)


In [29]:
config = {"configurable": {"thread_id": "chat-1"}}   # 👈 the conversation's name

In [35]:
response = agent.invoke({"messages":  (
        {"role": "user", "content": "Find the capital of Pakistan and then find its current weather and AQI."}
    )}, config)

/home/kashif/Desktop/learning/fde/agentic-ai/langchain-single-agents/.venv/lib/python3.11/site-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/home/kashif/Desktop/learning/fde/agentic-ai/langchain-single-agents/.venv/lib/python3.11/site-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


In [36]:
for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

Find the capital of India and then find its current weather and AQI.
================================== Ai Message ==================================

[]
Tool Calls:
  get_weather (call_17313066925248958665)
 Call ID: call_17313066925248958665
  Args:
    city: New Delhi
================================= Tool Message =================================
Name: get_weather

City: New Delhi
Temperature: 24°C
Weather: Smoky haze
Humidity: 52%
Air Quality (US EPA index): 5 - Very Unhealthy
PM2.5: 158.3 µg/m³
PM10: 195.1 µg/m³
================================== Ai Message ==================================

[{'type': 'text', 'text': 'The capital of India is **New Delhi**. \n\nHere is its current weather and air quality:\n* **Temperature:** 24°C\n* **Weather:** Smoky haze\n* **Humidity:** 52%\n* **Air Quality (US EPA index):** 5 - Very Unhealthy\n* **PM2.5:** 158.3 µg/m³\n* **PM10:** 195.1 µg/m³', 'extras': {'signa

In [37]:
response["messages"][-1].text

'The capital of Pakistan is **Islamabad**. \n\nHere is its current weather and air quality:\n* **Temperature:** 19°C\n* **Weather:** Smoky haze\n* **Humidity:** 53%\n* **Air Quality (US EPA index):** 4 - Unhealthy\n* **PM2.5:** 92.4 µg/m³\n* **PM10:** 105 µg/m³'

In [40]:
response2 = agent.invoke({"messages":  (
        {"role": "user", "content": "Whatever I asked earlier give me the answer."}
    )}, config)

/home/kashif/Desktop/learning/fde/agentic-ai/langchain-single-agents/.venv/lib/python3.11/site-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


In [41]:
response2["messages"][-1].text

'The capital of Pakistan is **Islamabad**, and its current weather is 19°C with smoky haze and an "Unhealthy" air quality index.'